In [6]:
import sqlite3
from pathlib import Path

In [12]:
pip install plotly pandas  numpy scipy scikit-learn statsmodels seaborn matplotlib


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------- -------------------------- 3.4/9.9 MB 28.6 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 31.8 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 25.7 MB/s  0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ----------------------------- ---------- 7.1/9.6 MB 31.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 27.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   ---------------------------------------- 0/3 [plotly]
   -----------------------


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\gunja\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\gunja\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import streamlit as st
import plotly.express as px



In [8]:
# ---------- Config ----------
DB_PATH = Path("C:/Users/gunja/Documents/Sqlite/mydb.db")  # adjust if different

st.set_page_config(
    page_title="UAC Care System Dashboard",
    layout="wide",
    initial_sidebar_state="expanded"
)

2026-05-30 22:28:11.193 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [9]:
@st.cache_data
def load_data():
    conn = sqlite3.connect(DB_PATH)
    df_feed = pd.read_sql("SELECT * FROM vw_uac_dashboard_feed", conn)
    df_kpi = pd.read_sql("SELECT * FROM vw_uac_kpi_daily", conn)
    df_month = pd.read_sql("SELECT * FROM vw_uac_monthly_summary", conn)
    conn.close()

    df_feed["report_date"] = pd.to_datetime(df_feed["report_date"])
    df_kpi["report_date"] = pd.to_datetime(df_kpi["report_date"])
    df_month["month_end_date"] = pd.to_datetime(df_month["month_end_date"])
    df_month["month_start_date"] = pd.to_datetime(df_month["month_start_date"])

    # If year / month not in views, compute here
    if "year" not in df_feed.columns:
        df_feed["year"] = df_feed["report_date"].dt.year
        df_feed["month"] = df_feed["report_date"].dt.month

    if "year" not in df_month.columns:
        df_month["year"] = df_month["month_end_date"].dt.year
        df_month["month"] = df_month["month_end_date"].dt.month

    return df_feed, df_kpi, df_month

df_feed, df_kpi, df_month = load_data()

2026-05-30 22:28:15.974 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 22:28:15.981 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-30 22:28:15.985 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:16.236 
  command:

    streamlit run C:\Users\gunja\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-30 22:28:16.236 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:16.237 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:16.237 Thread 'MainThread': missing ScriptRunContext! This warni

In [10]:
# ---------- Sidebar filters ----------
st.sidebar.title("Filters")

years = sorted(df_feed["year"].unique())
selected_years = st.sidebar.multiselect("Year", years, default=years)

df_feed = df_feed[df_feed["year"].isin(selected_years)]
df_kpi = df_kpi[df_kpi["report_date"].dt.year.isin(selected_years)]
df_month = df_month[df_month["year"].isin(selected_years)]

2026-05-30 22:28:20.889 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.890 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.891 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.892 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.893 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.894 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.895 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:20.895 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [11]:
# ---------- Executive Summary tab (business‑style) ----------

tab_exec, tab_ops, tab_risk, tab_scenario = st.tabs(
    ["Executive Summary", "Operations Monitor", "Backlog & Risk", "Scenario Lab"]
)

with tab_exec:
    st.header("Executive Summary")

    latest_date = df_feed["report_date"].max()
    latest = df_feed.loc[df_feed["report_date"] == latest_date].iloc[0]

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total System Load", f"{latest['total_system_load']:,}")
    c2.metric("Net Intake Pressure", f"{latest['net_intake_pressure']:,}")
    c3.metric("14‑Day Backlog", f"{latest['backlog_accumulation_14d']:,}")
    c4.metric("Discharge Offset Ratio", f"{latest['discharge_offset_ratio']:.2f}")

    # Monthly load comparison
    fig_month = px.bar(
        df_month.sort_values("month_end_date"),
        x="month_end_date",
        y="avg_total_system_load",
        color="year",
        title="Average Monthly System Load by Year",
        labels={"month_end_date": "Month", "avg_total_system_load": "Avg System Load"}
    )
    st.plotly_chart(fig_month, use_container_width=True)

    # Annual pressure
    annual_pressure = df_month.groupby("year", as_index=False)["total_net_intake_pressure"].sum()
    fig_pressure = px.bar(
        annual_pressure,
        x="year",
        y="total_net_intake_pressure",
        title="Annual Net Intake Pressure",
        labels={"total_net_intake_pressure": "Total Net Intake Pressure"}
    )
    st.plotly_chart(fig_pressure, use_container_width=True)

    # Short commentary
    if len(annual_pressure) >= 2:
        last = annual_pressure.sort_values("year").iloc[-1]
        prev = annual_pressure.sort_values("year").iloc[-2]
        delta = last["total_net_intake_pressure"] - prev["total_net_intake_pressure"]
        st.write(
            f"In {int(last['year'])}, net intake pressure was "
            f"{delta:+,} compared with {int(prev['year'])}, "
            "indicating whether the system is tightening or easing."
        )

2026-05-30 22:28:24.036 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.037 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.038 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.039 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.039 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.041 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:24.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [12]:
# ---------- Scenario Lab (simple what‑if) ----------

with tab_scenario:
    st.header("Scenario Lab")

    st.markdown(
        "Adjust assumptions for CBP transfers and HHS discharges to see how "
        "system load would evolve over the next 30 days."
    )

    col_a, col_b = st.columns(2)
    transfer_boost = col_a.slider("Increase CBP transfers out (%)", -20, 50, 10)
    discharge_boost = col_b.slider("Increase HHS discharges (%)", -20, 50, 10)

    recent = df_kpi.sort_values("report_date").tail(30).copy()
    factor_transfer = 1 + transfer_boost / 100
    factor_discharge = 1 + discharge_boost / 100

    recent["cbp_transferred_out_sim"] = recent["cbp_transferred_out"] * factor_transfer
    recent["hhs_discharged_sim"] = recent["hhs_discharged"] * factor_discharge

    # Very simple proxy: net_intake_pressure_sim
    recent["net_intake_pressure_sim"] = (
        recent["net_intake_pressure"]
        - (recent["hhs_discharged_sim"] - recent["hhs_discharged"])
    )

    fig_sim = px.line(
        recent,
        x="report_date",
        y=["net_intake_pressure", "net_intake_pressure_sim"],
        labels={"value": "Net Intake Pressure", "variable": "Scenario"},
        title="Baseline vs Scenario Net Intake Pressure (Last 30 Days Pattern)"
    )
    st.plotly_chart(fig_sim, use_container_width=True)

    st.caption(
        "This simple scenario assumes recent daily patterns continue. "
        "Higher discharges or transfers reduce future net intake pressure "
        "and help stabilise system load."
    )

2026-05-30 22:28:30.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.222 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.222 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.223 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 22:28:30.224 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar